In [10]:
import nflreadpy as nfl
import pandas as pd

# 1. Pull granular play-by-play data and convert to Pandas DataFrame
years = [2023, 2024, 2025]
pbp_df = nfl.load_pbp(years).to_pandas()

# 2. Filter for true offensive plays (removes kickoffs, punts, and penalties)
offensive_plays = pbp_df[pbp_df['play_type'].isin(['pass', 'run'])].dropna(subset=['epa', 'posteam', 'defteam'])

# 3. Calculate Offensive and Defensive EPA per play
off_weekly = offensive_plays.groupby(['posteam', 'season', 'week'])['epa'].mean().reset_index()
off_weekly.rename(columns={'posteam': 'team', 'epa': 'off_epa'}, inplace=True)

def_weekly = offensive_plays.groupby(['defteam', 'season', 'week'])['epa'].mean().reset_index()
def_weekly.rename(columns={'defteam': 'team', 'epa': 'def_epa'}, inplace=True)

# 4. Isolate pass plays to calculate Quarterback CPOE
pass_plays = offensive_plays[offensive_plays['play_type'] == 'pass'].dropna(subset=['cpoe'])
cpoe_weekly = pass_plays.groupby(['posteam', 'season', 'week'])['cpoe'].mean().reset_index()
cpoe_weekly.rename(columns={'posteam': 'team', 'cpoe': 'off_cpoe'}, inplace=True)

# 5. Merge all efficiency metrics into the new game log
team_game_log = pd.merge(off_weekly, def_weekly, on=['team', 'season', 'week'], how='inner')
team_game_log = pd.merge(team_game_log, cpoe_weekly, on=['team', 'season', 'week'], how='left')

# Calculate Net EPA per play as a single prioritizing metric
team_game_log['net_epa'] = team_game_log['off_epa'] - team_game_log['def_epa']

In [11]:
# 1. Sort chronologically
team_game_log = team_game_log.sort_values(by=['team', 'season', 'week'])

# 2. Define the rolling average function
def calculate_rolling_average(df, stat_column, window_size=3):
    return df.groupby('team')[stat_column].transform(
        lambda x: x.rolling(window=window_size, min_periods=1).mean().shift(1)
    )

# 3. Engineer the predictive efficiency features
team_game_log['rolling_3wk_off_epa'] = calculate_rolling_average(team_game_log, 'off_epa')
team_game_log['rolling_3wk_def_epa'] = calculate_rolling_average(team_game_log, 'def_epa')
team_game_log['rolling_3wk_net_epa'] = calculate_rolling_average(team_game_log, 'net_epa')
team_game_log['rolling_3wk_cpoe'] = calculate_rolling_average(team_game_log, 'off_cpoe')

# 4. Isolate the Colts and Texans to verify the shifts
afc_south_test = team_game_log[team_game_log['team'].isin(['IND', 'HOU'])].copy()
print(afc_south_test[['team', 'week', 'off_epa', 'rolling_3wk_off_epa', 'rolling_3wk_cpoe']].head(10))

    team  week   off_epa  rolling_3wk_off_epa  rolling_3wk_cpoe
639  HOU     1 -0.274915                  NaN               NaN
640  HOU     2 -0.015504            -0.274915         -4.569688
641  HOU     3  0.220129            -0.145210         -1.476859
642  HOU     4  0.140043            -0.023430          1.062058
643  HOU     5 -0.066452             0.114889          1.998035
644  HOU     6 -0.035225             0.097907         -1.116495
645  HOU     8 -0.034739             0.012789         -6.546529
646  HOU     9  0.298833            -0.045472         -6.317330
647  HOU    10  0.101376             0.076290          0.798145
648  HOU    11 -0.052880             0.121824          4.417289


In [12]:
import numpy as np

# 1. Load historical schedules for training data
years = [2023, 2024, 2025] 
sched_df = nfl.load_schedules(years).to_pandas()
sched_df = sched_df.dropna(subset=['home_score', 'away_score', 'spread_line'])

# 2. Calculate the actual game margin from the home team's perspective
sched_df['home_margin'] = sched_df['home_score'] - sched_df['away_score']

# 3. Map the binary outcomes (1 = Home Cover, 0 = Away Cover)
# nflreadpy uses positive numbers for home favorites and negative for away favorites.
conditions = [
    (sched_df['home_margin'] > sched_df['spread_line']),  # Home covered
    (sched_df['home_margin'] < sched_df['spread_line'])   # Away covered
]
choices = [1, 0]

sched_df['home_cover'] = np.select(conditions, choices, default=np.nan)
sched_df = sched_df.dropna(subset=['home_cover'])

# 4. Isolate the target variable dataframe
ml_target_df = sched_df[['season', 'week', 'home_team', 'away_team', 'spread_line', 'home_cover']]

In [13]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Merge the EPA and CPOE features to the Home Team
ml_target_df = pd.merge(
    ml_target_df, 
    team_game_log[['team', 'season', 'week', 'rolling_3wk_net_epa', 'rolling_3wk_cpoe']], 
    left_on=['home_team', 'season', 'week'], 
    right_on=['team', 'season', 'week'], 
    how='inner'
).rename(columns={
    'rolling_3wk_net_epa': 'home_net_epa', 
    'rolling_3wk_cpoe': 'home_cpoe'
}).drop(columns=['team'])

# 2. Merge the EPA and CPOE features to the Away Team
ml_target_df = pd.merge(
    ml_target_df, 
    team_game_log[['team', 'season', 'week', 'rolling_3wk_net_epa', 'rolling_3wk_cpoe']], 
    left_on=['away_team', 'season', 'week'], 
    right_on=['team', 'season', 'week'], 
    how='inner'
).rename(columns={
    'rolling_3wk_net_epa': 'away_net_epa', 
    'rolling_3wk_cpoe': 'away_cpoe'
}).drop(columns=['team'])

# 3. Clean incomplete rows (Week 1 NaNs generated by shifting)
complete_df = ml_target_df.dropna(subset=[
    'home_net_epa', 'home_cpoe', 'away_net_epa', 'away_cpoe', 'home_cover'
])

# 4. Strict Chronological Split (Train: 2023-2024, Test: 2025)
train_data = complete_df[complete_df['season'] < 2025]
test_data = complete_df[complete_df['season'] == 2025]

feature_cols = ['home_net_epa', 'home_cpoe', 'away_net_epa', 'away_cpoe']

X_train, y_train = train_data[feature_cols], train_data['home_cover']
X_test, y_test = test_data[feature_cols], test_data['home_cover']

# 5. Fit Random Forest & Evaluate (Using class_weight to fix the home bias)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

preds = rf_model.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, preds):.3f}")
print("Classification Report:\n", classification_report(y_test, preds))

Random Forest Accuracy: 0.553
Classification Report:
               precision    recall  f1-score   support

         0.0       0.56      0.44      0.49       140
         1.0       0.55      0.67      0.60       144

    accuracy                           0.55       284
   macro avg       0.55      0.55      0.55       284
weighted avg       0.55      0.55      0.55       284



In [14]:
# 1. Extract the probability arrays from the Random Forest model
# predict_proba returns a 2D array: [Probability of Class 0 (Away), Probability of Class 1 (Home)]
probabilities = rf_model.predict_proba(X_test)

# 2. Create a copy of the test data to avoid SettingWithCopy warnings
export_df = test_data.copy()

# 3. Append the hard predictions and the specific probabilities
export_df['predicted_cover'] = preds
export_df['prob_away_cover'] = probabilities[:, 0]
export_df['prob_home_cover'] = probabilities[:, 1]

# 4. Create a single "model_confidence" column (the max probability of the chosen bet)
export_df['model_confidence'] = export_df[['prob_away_cover', 'prob_home_cover']].max(axis=1)

# 5. Export directly to your local data folder (ignored by Git)
export_df.to_csv('../data/2025_nfl_predictions_with_confidence.csv', index=False)

# Display a preview of the high-confidence bets
print(export_df[['week', 'home_team', 'away_team', 'predicted_cover', 'model_confidence']].sort_values(by='model_confidence', ascending=False).head(10))

     week home_team away_team  predicted_cover  model_confidence
604     4       DET       CLE              1.0              0.90
673     9       MIA       BAL              1.0              0.89
739    13       TEN       JAX              1.0              0.88
763    15       JAX       NYJ              1.0              0.87
649     7        KC        LV              1.0              0.86
561     1       WAS       NYG              0.0              0.86
748    14       BUF       CIN              1.0              0.85
577     2       TEN        LA              0.0              0.83
678     9        NE       ATL              1.0              0.82
731    13       BAL       CIN              1.0              0.81
